# Bronze — Idempotent Batch Ingestion

Loads the 8 Olist CSV files from the landing volume into `globalmart.bronze.*` Delta tables.

**Design:** file fingerprint = `path + size + modificationTime`. If a fingerprint is already logged as `INGESTED`, the file is skipped. Each row is enriched with `_source_file_name`, `_source_path`, `_ingestion_run_id`, `_ingested_at`.

**Run this notebook twice** to prove idempotency — the second run should show `SKIPPED` and zero new rows.

In [ ]:
import sys

notebook_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)
if not notebook_path or "/notebooks/" not in notebook_path:
    raise ValueError("Run this notebook from the cloned repo under Databricks Repos.")

REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.ingestion.idempotent_loader import (
    IngestionConfig,
    ingest_landing_zone,
    build_ingestion_summary,
)

config = IngestionConfig(
    landing_path="/Volumes/globalmart/bronze/raw_landing"
)
print("Repo root (auto-detected):", REPO_ROOT)
print("Landing path:", config.landing_path)

In [ ]:
# Verify uploaded files
landing = config.landing_path
files = dbutils.fs.ls(landing)
display(spark.createDataFrame([(f.name, f.size) for f in files], ["file", "bytes"]))

In [ ]:
# First ingestion run
run_id, results = ingest_landing_zone(spark, dbutils, config)
print("Run ID:", run_id)
display(spark.createDataFrame([r.__dict__ for r in results]))

In [ ]:
# Summary report
from pyspark.sql import functions as F

summary = build_ingestion_summary(spark, config)
display(summary.orderBy(F.col("record_count").desc()))

top_records = summary.orderBy(F.col("record_count").desc()).first()
top_columns = summary.orderBy(F.col("column_count").desc()).first()
print(f"Most records: {top_records['bronze_table']} ({top_records['record_count']:,})")
print(f"Most columns:  {top_columns['bronze_table']} ({top_columns['column_count']})")

In [ ]:
# Idempotency proof — run ingestion again
run_id_2, results_2 = ingest_landing_zone(spark, dbutils, config)
display(spark.createDataFrame([r.__dict__ for r in results_2]))

skipped = sum(1 for r in results_2 if r.status == "SKIPPED")
ingested = sum(r.records_ingested for r in results_2)
print(f"Second run: {skipped} files skipped, {ingested} new records ingested")

In [ ]:
# Ingestion audit log
from pyspark.sql import functions as F

display(
    spark.table(config.metadata_table)
    .orderBy(F.col("ingested_at").desc())
)

In [ ]:
# Run summary — print JSON (copy to share). Optional save to catalog volume.
import json
from src.ingestion.idempotent_loader import build_run_report, save_run_report_to_volume

report = build_run_report(
    spark,
    config=config,
    first_run_id=run_id,
    first_results=results,
    second_run_id=run_id_2,
    second_results=results_2,
)

print(json.dumps(report, indent=2))

try:
    saved = save_run_report_to_volume(report, dbutils)
    print(f"\nSaved to catalog volume: {saved}")
except Exception as exc:
    print(f"\nVolume save skipped: {exc}")
    print("Re-run config/catalog_setup.sql to create globalmart.metadata.run_reports")